### Realtime Conversation with Voice Translation: Part 2 Spa to Eng
#### ` Description: Please create a machine learning model that facilitates real-time conversation between an English-speaking person and a Spanish-speaking person. The model should: Extract Spanish words from voice input and translate them into English, then read the translated word aloud. Similarly, take English voice input from the other user, translate it into Spanish, and read the translated word aloud. Guidelines: Make your own machine learning model. GUI is not mandatory for this. This task is a tough one. Don’t worry, accuracy doesn’t matter. The only thing matter is the amount of effort you have put. The evaluation will be conducted on models’ overall performance.`

## Extend the GPU memory 

In [5]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            
 
    except RuntimeError as e:
        print(e)


## Load all the libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
import nltk

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model,load_model
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu


## Load the dataset

In [2]:
df = pd.read_csv("eng_spa_small.csv")  
df.head(20)

,en,es
0,It's only three days till Christmas.,Solo faltan tres días para Navidad.
1,Harvard was founded in 1636.,Harvard se fundó en 1636.
2,Why do you waste most of your time on Tatoeba?,¿Por qué desperdicias la mayoría del tiempo en...
3,It is magnificent indeed.,Eso sí que es grandioso.
4,She traveled all over the world.,Ella viajó por todo el mundo.
5,Tom didn't mean to hurt you.,Tom no pensaba hacerte daño.
6,Tom asked for a loan from Mary.,Tom le pidió un préstamo a Mary.
7,"According to the long-term forecast, a mild wi...","Según el pronóstico del tiempo a largo plazo, ..."
8,That sounds interesting. What did you tell her?,Eso suena interesante. ¿Qué le dijiste?
9,Do you have any?,¿Tienes?


## Clean and preprocess the dataset

In [44]:
def clean_text(text):
    text = text.lower()
    text = text.replace("¿", "").replace("?", "")
    text = text.replace("¡", "").replace("!", "")
    text = text.replace(".", "").replace(",", "")
    return text.strip()

df['es'] = df['es'].apply(clean_text)
df['en'] = df['en'].apply(clean_text)

# Add start and end tokens to English 
df['en'] = df['en'].apply(lambda x: "<start> " + x + " <end>")


In [45]:
spa_tokenizer = Tokenizer(filters='')
spa_tokenizer.fit_on_texts(df['es'])

spa_sequences = spa_tokenizer.texts_to_sequences(df['es'])
max_spa_len = max(len(seq) for seq in spa_sequences)

encoder_input = pad_sequences(
    spa_sequences,
    maxlen=max_spa_len,
    padding='post'
)


In [46]:
eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(df['en'])

eng_sequences = eng_tokenizer.texts_to_sequences(df['en'])
max_eng_len = max(len(seq) for seq in eng_sequences)

decoder_input = pad_sequences(
    eng_sequences,
    maxlen=max_eng_len,
    padding='post'
)


In [7]:
decoder_target = np.zeros_like(decoder_input)

decoder_target[:, :-1] = decoder_input[:, 1:]
decoder_target[:, -1] = 0


## Build the Encoder-Decoder model

In [13]:
latent_dim = 256

encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(
    input_dim=len(spa_tokenizer.word_index) + 1,
    output_dim=latent_dim,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]


In [14]:
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(
    input_dim=len(eng_tokenizer.word_index) + 1,
    output_dim=latent_dim,
    mask_zero=True
)(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    len(eng_tokenizer.word_index) + 1,
    activation='softmax'
)

decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

## Compile and Train the model

In [67]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 embedding_2 (Embedding)        (None, None, 256)    3025664     ['input_1[0][0]']                
                                                                                                  
 embedding_3 (Embedding)        (None, None, 256)    2089984     ['input_2[0][0]']                
                                                                                              

In [ ]:
history = model.fit(
    [encoder_input, decoder_input],
    np.expand_dims(decoder_target, -1),
    batch_size=64,
    epochs=15, 
    validation_split=0.1
)


Epoch 1/15
211/211 [==============================] - 159s 631ms/step - loss: 2.2756 - accuracy: 0.2217 - val_loss: 2.0317 - val_accuracy: 0.2774
Epoch 2/15
211/211 [==============================] - 68s 322ms/step - loss: 1.9068 - accuracy: 0.2952 - val_loss: 1.9200 - val_accuracy: 0.3061
Epoch 3/15
211/211 [==============================] - 28s 134ms/step - loss: 1.7698 - accuracy: 0.3219 - val_loss: 1.8257 - val_accuracy: 0.3273
Epoch 4/15
211/211 [==============================] - 31s 149ms/step - loss: 1.6554 - accuracy: 0.3477 - val_loss: 1.7621 - val_accuracy: 0.3492
Epoch 5/15
211/211 [==============================] - 32s 152ms/step - loss: 1.5553 - accuracy: 0.3732 - val_loss: 1.7121 - val_accuracy: 0.3628
Epoch 6/15
211/211 [==============================] - 38s 180ms/step - loss: 1.4628 - accuracy: 0.3993 - val_loss: 1.6685 - val_accuracy: 0.3828
Epoch 7/15
211/211 [==============================] - 45s 212ms/step - loss: 1.3754 - accuracy: 0.4239 - val_loss: 1.6319 - val_a

### Run for more epochs to increase the accuracy

In [69]:
history = model.fit(
    [encoder_input, decoder_input],
    np.expand_dims(decoder_target, -1),
    batch_size=64,
    epochs=15,  
    validation_split=0.1
)


Epoch 1/15
211/211 [==============================] - 28s 130ms/step - loss: 0.7379 - accuracy: 0.6266 - val_loss: 1.5742 - val_accuracy: 0.4573
Epoch 2/15
211/211 [==============================] - 28s 133ms/step - loss: 0.6802 - accuracy: 0.6553 - val_loss: 1.5772 - val_accuracy: 0.4581
Epoch 3/15
211/211 [==============================] - 34s 161ms/step - loss: 0.6236 - accuracy: 0.6839 - val_loss: 1.5895 - val_accuracy: 0.4577
Epoch 4/15
211/211 [==============================] - 44s 207ms/step - loss: 0.5715 - accuracy: 0.7099 - val_loss: 1.6078 - val_accuracy: 0.4573
Epoch 5/15
211/211 [==============================] - 52s 246ms/step - loss: 0.5222 - accuracy: 0.7368 - val_loss: 1.6231 - val_accuracy: 0.4596
Epoch 6/15
211/211 [==============================] - 66s 311ms/step - loss: 0.4770 - accuracy: 0.7619 - val_loss: 1.6377 - val_accuracy: 0.4619
Epoch 7/15
211/211 [==============================] - 44s 207ms/step - loss: 0.4344 - accuracy: 0.7845 - val_loss: 1.6587 - val_ac

## Save the model and tokenizers

In [9]:
model.save("spa_to_eng_translation.h5")

import pickle

with open("spa_tokenizer_1.pkl", "wb") as f:
    pickle.dump(spa_tokenizer, f)

with open("eng_tokenizer_1.pkl", "wb") as f:
    pickle.dump(eng_tokenizer, f)


## Load the model and the tokenizers

In [71]:
model = load_model( "spa_to_eng_translation.h5")


with open("spa_tokenizer_1.pkl", "rb") as f:
    spa_tok_1 = pickle.load(f)

with open("eng_tokenizer_1.pkl", "rb") as f:
    eng_tok_1 = pickle.load(f)

with open("config_eng_spa.pkl", "rb") as f:
    config = pickle.load(f)

## Build an prediction function

In [79]:
def translate(text):
    seq = spa_tok_1.texts_to_sequences([text.lower()])
    seq = pad_sequences(seq, maxlen=max_spa_len, padding="post")

    decoder = [eng_tok_1.word_index["<start>"]]
    result = []

    for _ in range(max_eng_len):
        dec_seq = pad_sequences([decoder], maxlen=max_eng_len, padding="post")
        pred = model.predict([seq, dec_seq], verbose=0)

        idx = np.argmax(pred[0, len(decoder)-1])

        if idx == 0 or rev_eng.get(idx) == "<end>":
            break

        result.append(rev_eng[idx])
        decoder.append(idx)

    return " ".join(result)  


## Make some predictions

In [92]:
print("\n--- Testing Dataset Samples ---\n")

for i in np.random.choice(len(df), 5, replace=False):
    spanish = df.iloc[i]['es']
    actual_english = df.iloc[i]['en'].replace("<start>", "").replace("<end>", "")
    predicted = translate(df['es'][i])

    print("\nSpanish Input : ",spanish)
    print("Predicted Eng : ", predicted)
    print("Actual Eng    : ", actual_english)
  



--- Testing Dataset Samples ---


Spanish Input :  La decisión depende de ti.
Predicted Eng :  it's up to us to him
Actual Eng    :  It's up to you to decide.

Spanish Input :  Tú me pides que seamos solo amigos, y a mí no me interesa ser tu amigo.
Predicted Eng :  you may miss your favorite sentence to be able to do but i'm asking
Actual Eng    :  You're asking for us to just be friends, but I'm not interested in being your friend.

Spanish Input :  Tatoeba no es un diccionario.
Predicted Eng :  good is a liar
Actual Eng    :  Tatoeba isn't a dictionary.

Spanish Input :  Tengo mucho que aprender de ti.
Predicted Eng :  i have a lot to learn of people
Actual Eng    :  I have a lot to learn from you.

Spanish Input :  No puedo aceptar eso.
Predicted Eng :  i can't accept
Actual Eng    :  I can't accept that.
